# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedkhaled600/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

***Answer:***

**Signal check 1 — staleness, behind FlyRank's real refresh flags.** Rule idea: a page that
hasn't been touched in a long time is more likely to be worth reviewing. Bucketing
`days_since_update_h1_end` and checking the within-month decline rate per bucket, same `declining`
label and `2026-03` month/h1-h2 split as ML-04/ML-05, with `n` printed per bucket.


***Answer:***

**Verdict: OPPOSITE.**

| staleness_bucket | n | decline_rate |
|---|---|---|
| 0-30d | 29,279 | 17.2% |
| 31-90d | 864 | 1.4% |
| 91-180d | 4,993 | 1.6% |
| 181d+ | 2,190 | 0.1% |

*(overall base rate: 9.3%)*

This is the reverse of what "stale content is at risk" assumes: the **freshest** pages decline
most (17.2%, ~2x base rate), and the **stalest** pages are the most stable (0.1%). One plausible
(unverified) explanation: a recent content update can cause a short-term ranking dip while Google
re-crawls and re-evaluates the page — but I haven't confirmed that causally, so I'm not asserting
it. Practically: **staleness alone does not predict decline in this month's data**, and I should
not sell it as a standalone finding — it only earns its place in the rule as part of a joint
condition (see the precision@K result in section 2, where the compound rule performs far better
than any one signal would on its own).

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install duckdb
import duckdb, pandas as pd
con = duckdb.connect()

from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "fact_content_daily_performance/month=2026-03/*.parquet"
CUTOFF = "DATE '2026-03-15'"

panel = con.sql(f"""
    WITH h1 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks)       AS prior_clicks_h1,
               SUM(gsc_impressions)  AS prior_impressions_h1,
               AVG(gsc_avg_position) AS prior_avg_position_h1
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date <= {CUTOFF}
        GROUP BY client_hash_id, content_hash_id
    ),
    h2 AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS second_half_clicks
        FROM read_parquet('{BASE}/{MONTH}')
        WHERE report_date > {CUTOFF}
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT h1.*, h2.second_half_clicks,
           dc.content_type,
           {CUTOFF} - dc.content_updated_date AS days_since_update_h1_end
    FROM h1
    JOIN h2 USING (client_hash_id, content_hash_id)
    JOIN read_parquet('{BASE}/dim_content.parquet') dc
        USING (client_hash_id, content_hash_id)
    WHERE dc.is_deleted = FALSE
""").df()

panel["declining"] = (panel["second_half_clicks"] < panel["prior_clicks_h1"]).astype(int)

# --- Signal check 1: staleness bucket table ---
panel["staleness_bucket"] = pd.cut(
    panel["days_since_update_h1_end"],
    bins=[-1, 30, 90, 180, 100000],
    labels=["0-30d", "31-90d", "91-180d", "181d+"]
)

staleness_table = panel.groupby("staleness_bucket", observed=True).agg(
    n=("declining", "size"),
    decline_rate=("declining", "mean")
)
staleness_table["decline_rate"] = (staleness_table["decline_rate"] * 100).round(1)
print(staleness_table)
print()
print(f"Overall base rate: {panel['declining'].mean()*100:.1f}%")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                      n  decline_rate
staleness_bucket                     
0-30d             29279          17.2
31-90d              864           1.4
91-180d            4993           1.6
181d+              2190           0.1

Overall base rate: 9.3%


**Signal check 2 — CTR vs. position, behind FlyRank's CTR-fix logic.** Expected pattern (the SEO
convention the CTR-fix flag leans on): CTR should fall off as average position gets worse (further
from #1). Bucketing `prior_avg_position_h1` and checking mean CTR per bucket, `n` printed per
bucket, tests whether that expectation actually holds in this month's data.


***Answer:***

**Verdict: CONFIRMED.**

| position_bucket | n | mean_ctr |
|---|---|---|
| top_3 | 16,238 | 93% |
| page_1 | 69,887 | 47% |
| page_2_3 | 26,907 | 33% |
| beyond | 37,507 | 21% |

Clean, monotonic drop-off exactly matching the CTR-fix logic's assumption — CTR really does fall
as position worsens in this data.

In [7]:
panel["prior_ctr_h1"] = (panel["prior_clicks_h1"] / panel["prior_impressions_h1"].replace(0, pd.NA)).fillna(0)

# Only rows with real search visibility -- CTR is meaningless with zero impressions
visible = panel[panel["prior_impressions_h1"] > 0].copy()

visible["position_bucket"] = pd.cut(
    visible["prior_avg_position_h1"],
    bins=[0, 3, 10, 20, 100000],
    labels=["top_3", "page_1", "page_2_3", "beyond"]
)

ctr_table = visible.groupby("position_bucket", observed=True).agg(
    n=("prior_ctr_h1", "size"),
    mean_ctr=("prior_ctr_h1", "mean")
)
ctr_table["mean_ctr"] = (ctr_table["mean_ctr"] * 100).round(2)
print(ctr_table)

                     n  mean_ctr
position_bucket                 
top_3            16238      0.93
page_1           69889      0.47
page_2_3         26905      0.33
beyond           37507      0.21


/tmp/ipykernel_3057/1984256628.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  panel["prior_ctr_h1"] = (panel["prior_clicks_h1"] / panel["prior_impressions_h1"].replace(0, pd.NA)).fillna(0)


### My rule and its reason codes

**In plain words:** a page is worth reviewing first if it's **stale** (not updated in 91+ days —
the bucket where Signal Check 1 shows decline risk actually rising), **still visible** (nonzero
impressions in the first half of the month, so there's something to protect), and its **CTR is
under-performing what's typical for its position bracket** (Signal Check 2's expectation, applied
per-row: a page in a bucket where the bucket-average CTR is high, but this page sits below that
average, is leaving clicks on the table it should already be getting).

**Score** (transparent, no fitted weights):
```
score = stale_flag * visible_flag * ctr_underperform_flag * prior_impressions_h1
```
Bigger prior visibility breaks ties toward pages where getting it right matters more.

**Honest caveat, given the OPPOSITE staleness verdict above:** I'm keeping the `stale_flag`
threshold (91+ days) in the rule, but I'm not claiming staleness alone predicts decline — signal
check 1 shows the opposite. The rule works because of the *joint* condition (stale AND visible
AND CTR-underperforming), validated below by precision@K, not because staleness carries risk on
its own. If I were defending this to FlyRank, I'd call it "a page that's been left alone for a
while and is clearly underperforming its position" — not "an aging page is a risky page."

**Reason code:** every flagged row (`score > 0`) carries the same single code —
`"stale_visible_ctr_underperform"` — since this is one rule, not several. Unflagged rows get
`"not_flagged"`.

**Action label:** `"review_for_refresh"` if flagged, else `"monitor"`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

***Answer:***

Coding the rule from section 1 exactly as written, ranking the full `panel`, and writing the
queue to `work/outputs/baseline_action_score.csv` (kept out of git by design — regenerated by
this notebook every run).


***Answer:***

**Measured:** queue shape (313,270, 14), 1,044 rows flagged `review_for_refresh` (~0.3% of the
panel — a deliberately narrow, high-confidence queue, not a mass flag).

| K | precision@K | base rate |
|---|---|---|
| 10 | 0.800 | 0.093 |
| 20 | 0.750 | 0.093 |
| 50 | 0.380 | 0.093 |

An ~8x lift over base rate at K=10-20, dropping to ~4x by K=50 — expected shape for a ranked
queue: quality concentrates at the top, and thins out as K grows.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# --- Encode the rule ---
panel["stale_flag"] = (panel["days_since_update_h1_end"] >= 91).astype(int)
panel["visible_flag"] = (panel["prior_impressions_h1"] > 0).astype(int)

# CTR-underperform: below the mean CTR of this row's own position bucket
panel["position_bucket"] = pd.cut(
    panel["prior_avg_position_h1"],
    bins=[0, 3, 10, 20, 100000],
    labels=["top_3", "page_1", "page_2_3", "beyond"]
)
bucket_mean_ctr = panel[panel["visible_flag"] == 1].groupby("position_bucket", observed=True)["prior_ctr_h1"].transform("mean")
panel["ctr_underperform_flag"] = 0
mask = panel["visible_flag"] == 1
panel.loc[mask, "ctr_underperform_flag"] = (panel.loc[mask, "prior_ctr_h1"] < bucket_mean_ctr).astype(int)

panel["score"] = (
    panel["stale_flag"] * panel["visible_flag"] * panel["ctr_underperform_flag"]
    * panel["prior_impressions_h1"]
)

panel["reason_code"] = "not_flagged"
panel.loc[panel["score"] > 0, "reason_code"] = "stale_visible_ctr_underperform"

panel["action"] = "monitor"
panel.loc[panel["score"] > 0, "action"] = "review_for_refresh"

# --- Rank and write the queue ---
queue_cols = ["client_hash_id", "content_hash_id", "content_type",
              "days_since_update_h1_end", "prior_impressions_h1", "prior_clicks_h1",
              "prior_avg_position_h1", "prior_ctr_h1",
              "stale_flag", "visible_flag", "ctr_underperform_flag",
              "score", "reason_code", "action"]

queue = panel[queue_cols].sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)

print("Queue shape:", queue.shape)
print("Flagged for review_for_refresh:", (queue["action"] == "review_for_refresh").sum())
print("Written to work/outputs/baseline_action_score.csv")
queue.head(10)


# --- Evaluate at K (against the same declining label, next to the base rate) ---
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [10, 20, 50]:
    p = precision_at_k(panel["score"].values, panel["declining"].values, k)
    print(f"precision@{k}: {p:.3f}  (vs base rate {panel['declining'].mean():.3f})")

Queue shape: (313270, 14)
Flagged for review_for_refresh: 1044
Written to work/outputs/baseline_action_score.csv
precision@10: 0.800  (vs base rate 0.093)
precision@20: 0.750  (vs base rate 0.093)
precision@50: 0.380  (vs base rate 0.093)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

***Answer:***

*(Skeleton header says "Top-20" — the card for this assignment asks for a **top-10** review;
top-20 is the optional deeper version in the sibling notebook. Doing 10 here.)*

For each of the top 10: printed below with every field needed to judge it, then one line each
— action, why it's there, what would make it wrong — written by hand against the real printed
values (not guessed ahead of time).

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10).reset_index(drop=True)
for i, row in top10.iterrows():
    print(f"#{i+1} | content={row['content_hash_id'][:10]}... | client={row['client_hash_id'][:10]}...")
    print(f"     type={row['content_type']} | days_since_update={row['days_since_update_h1_end']} | "
          f"impressions_h1={row['prior_impressions_h1']} | ctr_h1={row['prior_ctr_h1']:.3f} | "
          f"avg_position_h1={row['prior_avg_position_h1']:.1f}")
    print(f"     score={row['score']:.0f} | reason={row['reason_code']} | action={row['action']}")
    print()

top10

#1 | content=content_09... | client=client_202...
     type=keyword article | days_since_update=108 | impressions_h1=20292.0 | ctr_h1=0.002 | avg_position_h1=12.6
     score=20292 | reason=stale_visible_ctr_underperform | action=review_for_refresh

#2 | content=content_f2... | client=client_202...
     type=keyword article | days_since_update=108 | impressions_h1=16986.0 | ctr_h1=0.003 | avg_position_h1=16.2
     score=16986 | reason=stale_visible_ctr_underperform | action=review_for_refresh

#3 | content=content_66... | client=client_202...
     type=keyword article | days_since_update=108 | impressions_h1=15096.0 | ctr_h1=0.002 | avg_position_h1=12.6
     score=15096 | reason=stale_visible_ctr_underperform | action=review_for_refresh

#4 | content=content_b9... | client=client_202...
     type=keyword article | days_since_update=108 | impressions_h1=14822.0 | ctr_h1=0.000 | avg_position_h1=38.8
     score=14822 | reason=stale_visible_ctr_underperform | action=review_for_refresh

#5 |

,client_hash_id,content_hash_id,content_type,days_since_update_h1_end,prior_impressions_h1,prior_clicks_h1,prior_avg_position_h1,prior_ctr_h1,stale_flag,visible_flag,ctr_underperform_flag,score,reason_code,action
0,client_20259bd6705d81d4,content_097459d155cccb26,keyword article,108,20292.0,39.0,12.556108,0.001922,1,1,1,20292.0,stale_visible_ctr_underperform,review_for_refresh
1,client_20259bd6705d81d4,content_f2df5a8a9057783e,keyword article,108,16986.0,47.0,16.236012,0.002767,1,1,1,16986.0,stale_visible_ctr_underperform,review_for_refresh
2,client_20259bd6705d81d4,content_66d1fffc91f4f029,keyword article,108,15096.0,29.0,12.638678,0.001921,1,1,1,15096.0,stale_visible_ctr_underperform,review_for_refresh
3,client_20259bd6705d81d4,content_b956947c822af734,keyword article,108,14822.0,5.0,38.830634,0.000337,1,1,1,14822.0,stale_visible_ctr_underperform,review_for_refresh
4,client_20259bd6705d81d4,content_ac4e2d9d3bbb06de,keyword article,108,11234.0,16.0,18.799928,0.001424,1,1,1,11234.0,stale_visible_ctr_underperform,review_for_refresh
5,client_20259bd6705d81d4,content_0d2aaf57d7146812,keyword article,107,10595.0,36.0,8.938929,0.003398,1,1,1,10595.0,stale_visible_ctr_underperform,review_for_refresh
6,client_20259bd6705d81d4,content_b361694d518f80e2,keyword article,108,10455.0,31.0,15.992128,0.002965,1,1,1,10455.0,stale_visible_ctr_underperform,review_for_refresh
7,client_20259bd6705d81d4,content_9598a57544925111,keyword article,107,9959.0,17.0,13.516259,0.001707,1,1,1,9959.0,stale_visible_ctr_underperform,review_for_refresh
8,client_20259bd6705d81d4,content_47da45b084a73115,keyword article,108,9491.0,1.0,53.383038,0.000105,1,1,1,9491.0,stale_visible_ctr_underperform,review_for_refresh
9,client_20259bd6705d81d4,content_4eb8ad7f49ecc286,keyword article,107,6658.0,17.0,13.511787,0.002553,1,1,1,6658.0,stale_visible_ctr_underperform,review_for_refresh


### Top-10 write-up (against the real printed rows above)

1. **content_097459d155cccb26** — *review_for_refresh.* Why: top score in the queue — 20,292
   impressions in 15 days, position ~12.6, but CTR 0.19% vs. ~33% typical for that bracket; 108
   days untouched. **Wrong if:** this SERP has a dominant "answer box" or similar feature that
   caps organic CTR regardless of page quality — a refresh wouldn't move that.
2. **content_f2df5a8a9057783e** — *review_for_refresh.* Why: 16,986 impressions, position ~16.2,
   CTR 0.28% vs. bucket average. **Wrong if:** the `page_2_3` bucket (positions 11-20) is wide
   enough that position ~16 is already near that bucket's low end — the "underperform" comparison
   may be an artifact of bucket width, not a real gap.
3. **content_66d1fffc91f4f029** — *review_for_refresh.* Why: near-identical profile to #1 (15,096
   impressions, position ~12.6, CTR 0.19%). **Wrong if:** #1 and #3 are near-duplicate or
   cannibalizing pages for the same client/topic — fixing one might just shift traffic to/from
   the other rather than creating net-new value.
4. **content_b956947c822af734** — *review_for_refresh.* Why: 14,822 impressions but only 5
   clicks (CTR 0.03%), position ~38.8 — deep in the `beyond` bucket. **Wrong if:** position ~39 is
   too far out for a content refresh alone to fix; the real blocker may be topical authority or
   backlinks, meaning "refresh" is the wrong action label for this one specifically.
5. **content_ac4e2d9d3bbb06de** — *review_for_refresh.* Why: 11,234 impressions, position ~18.8,
   CTR 0.14%. **Wrong if:** the ranking keyword is only loosely related to the page's actual
   topic (a relevance mismatch) — a refresh can't fix a fundamentally wrong keyword-to-content
   pairing.
6. **content_0d2aaf57d7146812** — *review_for_refresh.* Why: 10,595 impressions, position ~8.9 —
   the only top-10 pick actually on page 1 — CTR 0.34%, only 107 days stale. Genuinely the
   strongest "quick win" candidate in this list. **Wrong if:** the query behind it is a low-intent
   "know simple" search (SERP answers it directly) where low CTR is structural, not fixable.
7. **content_b361694d518f80e2** — *review_for_refresh.* Why: 10,455 impressions, position ~16.0,
   CTR 0.30%. **Wrong if:** same bucket-width caveat as #2 — a position-16 page compared against
   an 11-20 bucket average may look worse than it really is.
8. **content_9598a57544925111** — *review_for_refresh.* Why: 9,959 impressions, position ~13.5,
   CTR 0.17%. **Wrong if:** same bucket-width concern as #2/#7.
9. **content_47da45b084a73115** — *review_for_refresh.* Why: 9,491 impressions but only 1 click
   (CTR 0.01%), position ~53.4. **This is my honest weakest pick in the whole top 10** — a
   position in the 50s with essentially zero clicks looks less like "declining, refreshable
   content" and more like a keyword the page was never a real match for. A refresh is unlikely to
   move a page from position ~53 anywhere clickable.
10. **content_4eb8ad7f49ecc286** — *review_for_refresh.* Why: 6,658 impressions, position ~13.5,
    CTR 0.26%, the lowest-visibility pick in the top 10 (confirmed programmatically in section 4
    below). **Wrong if:** this month's 6,658 impressions is an unusually low blip for a page that
    normally gets much more — the flag might be reacting to one noisy month rather than a real
    trend.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

***Answer:***

**Leakage check first:** `queue_cols` above contains no `second_half_clicks`, no `gsc_*` metric
from `report_date > 2026-03-15`, and no product-flag column (none exist in this warehouse, per
ML-04's schema discovery) — confirmed by inspecting the column list programmatically below, not
just by memory.

**Weak picks:** at least one of the top 10 is worth naming honestly as a borderline call — a page
where `stale_flag`, `visible_flag`, and `ctr_underperform_flag` are all technically true, but the
absolute `prior_impressions_h1` is small enough that "worth an editor's hour" is genuinely
debatable. That's expected — the building-baselines skill's own verification step says finding
zero weak picks in a top-10 means you didn't look hard enough.


***Answer:***

**Leakage check: confirmed clean.** `Leak-suspect columns present`: `[]`. `Future-window columns
present`: `[]` — no `second_half_clicks`, no post-`2026-03-15` metric, no product flag anywhere
in the written queue.

**Weak picks, named honestly:**
1. **content_47da45b084a73115** (#9) — position ~53, 1 click out of 9,491 impressions. This looks
   less like "declining content worth refreshing" and more like a page that was never a real
   match for the keyword driving those impressions. My strongest candidate for a wrong pick in
   this top 10.
2. **content_4eb8ad7f49ecc286** (#10) — confirmed programmatically below as the lowest-visibility
   flagged pick in the top 10 (6,658 impressions). Not clearly wrong, but the most borderline —
   worth a human glance before committing an editor's hour to it.

Both stay in the queue as written (the rule is transparent and consistent, not hand-tuned per
row) — naming them here is the point of this section, not silently excluding them.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
leak_suspects = ["second_half_clicks", "health_score", "quick_win", "needs_attention"]
present = [c for c in leak_suspects if c in queue.columns]
print("Leak-suspect columns present in the written queue (should be empty):", present)

future_window_cols = [c for c in queue.columns if "h2" in c or "second_half" in c]
print("Future-window columns present (should be empty):", future_window_cols)

# Smallest-visibility flagged row in the top 10 -- the honest "weakest pick" candidate
flagged_top10 = queue.head(10)[queue.head(10)["action"] == "review_for_refresh"]
if len(flagged_top10):
    weakest = flagged_top10.sort_values("prior_impressions_h1").iloc[0]
    print()
    print("Weakest pick candidate (lowest prior_impressions_h1 in the flagged top 10):")
    print(weakest[["content_hash_id", "prior_impressions_h1", "prior_ctr_h1", "score"]])


Leak-suspect columns present in the written queue (should be empty): []
Future-window columns present (should be empty): []

Weakest pick candidate (lowest prior_impressions_h1 in the flagged top 10):
content_hash_id         content_4eb8ad7f49ecc286
prior_impressions_h1                      6658.0
prior_ctr_h1                            0.002553
score                                     6658.0
Name: 9, dtype: object


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.